# Healthcare AI RAG System - Complete Implementation

## Project: Build Your Own Project (BYOP) - Healthcare AI
### Author: Saumy DhoLu
### Date: 29/08/2025

### Overview
This notebook implements a complete Retrieval-Augmented Generation (RAG) system from scratch. The goal is to answer complex healthcare questions by leveraging a large dataset of medical research papers from PubMed. The system is designed with a modular, three-layer architecture: Embedding, Search, and Generation.

### =============================================================================
# SECTION 1: IMPORTS AND SETUP
### =============================================================================

First things first, let's get all our tools ready. We're importing all the necessary libraries for data handling (Pandas, NumPy), NLP and machine learning (SentenceTransformers, FAISS, Scikit-learn), interacting with the Gemini API, and for creating visualizations.

In [4]:
# Core libraries for data handling and system operations
import pandas as pd
import numpy as np
import pickle
import json
import os
import re
import time
import csv
from datetime import datetime
import warnings
warnings.filterwarnings('ignore') # Let's keep the output clean

# Machine Learning and NLP heavy-hitters
from sentence_transformers import SentenceTransformer, CrossEncoder # For creating and re-ranking embeddings
import faiss # Facebook's lightning-fast similarity search library
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# The star of the show: Google Gemini AI
import google.generativeai as genai

# Visualization and progress bars to make things look nice
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets
from tqdm.notebook import tqdm

print("All required libraries imported successfully!")


All required libraries imported successfully!


### =============================================================================
# SECTION 2: CONFIGURATION AND DATA LOADING
### =============================================================================


Now, let's load our configuration (like the API key) and the main dataset of medical papers. It's crucial to handle this properly so the rest of the system can run smoothly.

In [7]:
def load_configuration():
    """Loads the Gemini API key from a local file and configures the generative model.
    
    This function keeps the API key separate from the code for better security.
    """
    try:
        # The API key should be in a file named 'GEMINI_API_KEY.txt' in the same directory.
        with open('GEMINI_API_KEY.txt', 'r') as f:
            api_key = f.read().strip()
        
        # Configure the Gemini API with the key
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel('gemini-2.5-pro') # We'll use the powerful 2.5 Pro model
        print("Gemini API configured successfully.")
        return model
    
    except FileNotFoundError:
        print("Error: 'GEMINI_API_KEY.txt' not found. Please create this file and add your API key.")
        return None
    except Exception as e:
        print(f"An error occurred while loading the API key: {e}")
        return None

In [8]:
def load_healthcare_dataset():
    """Loads the medical research papers from the CSV file.
    
    It also performs some initial inspection like checking the shape, columns, and missing values.
    """
    filename = 'healthcare_papers_20250827_102302.csv'
    
    try:
        # Using pandas to load our dataset into a DataFrame
        df = pd.read_csv(filename)
        print(f"Dataset '{filename}' loaded successfully with {len(df)} papers.")
        
        # Convert the publication date to a proper datetime format for later analysis
        df['publication_date'] = pd.to_datetime(df['publication_date'], errors='coerce')
        
        # Let's get a quick overview of our data
        print(f"\nDataset Info:")
        print(f"  - Shape: {df.shape}")
        print(f"  - Columns: {list(df.columns)}")
        
        # Checking for any missing data is a good first step in any data project
        missing_counts = df.isnull().sum()
        if missing_counts.sum() > 0:
            print("\nMissing values per column:")
            print(missing_counts[missing_counts > 0])
        
        # Basic statistics to understand the dataset's scope
        print(f"\nBasic Stats:")
        if df['publication_date'].notnull().any():
            print(f"  - Date range: {df['publication_date'].min().date()} to {df['publication_date'].max().date()}")
        print(f"  - Average abstract word count: {df['word_count'].mean():.0f}")
        print(f"  - Number of medical categories: {df['condition_category'].nunique()}")
        
        return df
    
    except FileNotFoundError:
        print(f"Error: Dataset file '{filename}' not found. Make sure it's in the correct directory.")
        return None
    except Exception as e:
        print(f"An error occurred while loading the dataset: {e}")
        return None

In [9]:
# Let's initialize the system by loading everything.
gemini_model = load_configuration()
healthcare_df = load_healthcare_dataset()

# A quick check to make sure we can proceed. No API key or data, no RAG system!
if healthcare_df is None or gemini_model is None:
    print("\nExecution stopped. Please resolve the errors above before continuing.")
    # exit() # Uncomment this line if running as a script to halt execution

Gemini API configured successfully.
Dataset 'healthcare_papers_20250827_102302.csv' loaded successfully with 60380 papers.

Dataset Info:
  - Shape: (60380, 12)
  - Columns: ['pmid', 'title', 'abstract', 'authors', 'journal', 'publication_date', 'url', 'full_text', 'word_count', 'collection_date', 'medical_condition', 'condition_category']

Missing values per column:
authors             202
publication_date    325
dtype: int64

Basic Stats:
  - Date range: 2019-03-01 to 2026-06-01
  - Average abstract word count: 262
  - Number of medical categories: 22


### =============================================================================
# SECTION 3: TEXT PREPROCESSING & MEDICAL TERMINOLOGY HANDLING
### =============================================================================


This is the first and most critical step in the **Embedding Layer**. Medical text is messy. To make it useful for an AI model, we need to clean it up and handle domain-specific language. We'll create a dedicated class for this to keep our logic clean and organized.

In [12]:
class MedicalTextProcessor:
    """A class to handle all text preprocessing, with a special focus on medical terminology.
    
    This is a key part of our RAG system's 'secret sauce'. Better text means better search.
    """
    
    def __init__(self):
        # A hand-curated dictionary to expand common medical abbreviations.
        # This improves semantic search, as 'MI' and 'myocardial infarction' will be treated similarly.
        self.medical_abbreviations = {
            'MI': 'myocardial infarction', 'DM': 'diabetes mellitus', 'T2DM': 'type 2 diabetes mellitus',
            'HTN': 'hypertension', 'CHF': 'congestive heart failure', 'COPD': 'chronic obstructive pulmonary disease',
            'CAD': 'coronary artery disease', 'CVD': 'cardiovascular disease', 'HbA1c': 'hemoglobin A1c',
            'BP': 'blood pressure', 'HR': 'heart rate', 'BMI': 'body mass index', 'ICU': 'intensive care unit',
            'ER': 'emergency room', 'IV': 'intravenous', 'PO': 'oral administration', 'BID': 'twice daily',
            'TID': 'three times daily', 'QID': 'four times daily', 'PRN': 'as needed', 'ACE': 'angiotensin converting enzyme',
            'ARB': 'angiotensin receptor blocker', 'CCB': 'calcium channel blocker', 'NSAID': 'nonsteroidal anti-inflammatory drug',
            'SSRI': 'selective serotonin reuptake inhibitor'
        }
        
        # Pre-compiling regex patterns makes the text expansion much faster.
        # We're looking for whole words only (using \b for word boundaries) to avoid replacing parts of other words.
        self.abbr_patterns = {
            abbr: re.compile(rf'\b{re.escape(abbr)}\b', re.IGNORECASE)
            for abbr in self.medical_abbreviations
        }
    
    def expand_medical_abbreviations(self, text):
        """Expands medical abbreviations found in the text.
        
        For example, 'patient with MI' -> 'patient with MI (myocardial infarction)'
        We keep both the abbreviation and the full form to maximize matching potential.
        """
        if not isinstance(text, str):
            return "" # Handle potential NaN or non-string values gracefully
        
        for abbr, pattern in self.abbr_patterns.items():
            full_form = self.medical_abbreviations[abbr]
            text = pattern.sub(f"{abbr} ({full_form})", text)
        
        return text
    
    def clean_medical_text(self, text):
        """A general-purpose cleaning function for medical text.
        
        This performs abbreviation expansion, whitespace normalization, and removes unwanted characters.
        """
        if not isinstance(text, str):
            return ""
        
        # Step 1: Expand abbreviations to add semantic context
        text = self.expand_medical_abbreviations(text)
        
        # Step 2: Normalize all whitespace to single spaces
        text = ' '.join(text.split())
        
        # Step 3: Remove most special characters, but keep ones that are medically relevant (e.g., hyphens, parentheses)
        text = re.sub(r'[^\w\s\-\.\(\)\,\:\;]', ' ', text)
        
        return text.strip()
    
    def process_dataframe(self, df):
        """Applies all preprocessing steps to the main DataFrame.
        
        This creates new columns with the processed text, keeping the original data intact.
        It also creates a final 'searchable_content' column that combines title and abstract for embedding.
        """
        print("Processing medical text data...")
        
        # Apply our cleaning function to the key text fields
        df['title_processed'] = df['title'].apply(self.clean_medical_text)
        df['abstract_processed'] = df['abstract'].apply(self.clean_medical_text)
        df['full_text_processed'] = df['full_text'].apply(self.clean_medical_text)
        
        # Combining title and abstract is a common strategy to create a rich, searchable text chunk.
        df['searchable_content'] = df['title_processed'] + " " + df['abstract_processed']
        
        print("Text processing complete!")
        return df

In [13]:
# Create an instance of our processor and apply it to the dataframe.
text_processor = MedicalTextProcessor()
healthcare_df = text_processor.process_dataframe(healthcare_df)

# Let's check a sample to see if our processing worked as expected.
print("\nSample processed text:")
print(healthcare_df['searchable_content'].iloc[0][:200] + "...")

Processing medical text data...
Text processing complete!

Sample processed text:
Single-Dose Psilocybin for a Treatment-Resistant Episode of Major Depression. Psilocybin is being studied for use in treatment-resistant depression. In this phase 2 double-blind trial, we randomly ass...


### =============================================================================
# SECTION 4: CHUNKING STRATEGIES IMPLEMENTATION & COMPARISON
### =============================================================================

The second part of the **Embedding Layer**. How we split our documents into smaller pieces (chunks) has a huge impact on retrieval quality. Here, we'll implement several strategies and compare them statistically to make an informed decision, rather than just guessing.

In [16]:
class ChunkingStrategyAnalyzer:
    """A class to implement and compare different text chunking strategies.
    
    This allows us to experiment and choose the best way to split our documents for this specific dataset.
    """
    
    def __init__(self, dataframe):
        self.df = dataframe
        self.chunking_results = {} # To store results for comparison
    
    # --- STRATEGY 1: Sentence-Based Chunking ---
    def sentence_based_chunking(self, text, sentences_per_chunk=3):
        """Splits text by sentence and groups them into chunks.
        Good for maintaining sentence structure, but can result in uneven chunk lengths.
        """
        if not isinstance(text, str) or not text.strip():
            return []
        
        sentences = re.split(r'[.!?]+', text) # Split by common sentence terminators
        sentences = [s.strip() for s in sentences if s.strip()]
        
        chunks = ['. '.join(sentences[i:i + sentences_per_chunk]).strip() + '.' 
                  for i in range(0, len(sentences), sentences_per_chunk)]
        return [c for c in chunks if c.strip() and c != '.'] # Filter out empty chunks

    # --- STRATEGY 2: Fixed-Length Chunking ---
    def fixed_length_chunking(self, text, chunk_size=500, overlap=50):
        """Splits text into fixed-size chunks with some overlap.
        Simple and fast, but risks cutting sentences and losing context.
        Overlap helps mitigate the context loss at the edges.
        """
        if not isinstance(text, str) or not text.strip():
            return []
        
        chunks = []
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk = text[start:end].strip()
            if chunk:
                chunks.append(chunk)
            start += chunk_size - overlap
        return chunks

    # --- STRATEGY 3: Semantic Chunking (Our Preferred Method) ---
    def semantic_chunking(self, text, max_chunk_size=400):
        """Chunks based on semantic boundaries (sentences) but with a length limit.
        This is often the best of both worlds: maintains context while controlling size.
        """
        if not isinstance(text, str) or not text.strip():
            return []
        
        sentences = re.split(r'[.!?]+', text)
        sentences = [s.strip() for s in sentences if s.strip()]
        
        chunks = []
        current_chunk = ""
        for sentence in sentences:
            if len(current_chunk) + len(sentence) + 1 > max_chunk_size and current_chunk:
                chunks.append(current_chunk.strip() + '.')
                current_chunk = sentence
            else:
                current_chunk += ' ' + sentence if current_chunk else sentence
        
        if current_chunk.strip():
            chunks.append(current_chunk.strip() + '.')
            
        return chunks
    
    def apply_chunking_strategy(self, strategy_name, sample_size=1000, **kwargs):
        """Applies a chunking strategy to a sample of the dataset and collects stats.
        Running on a sample is faster for analysis.
        """
        print(f"Applying '{strategy_name}' chunking strategy...")
        sample_df = self.df.head(sample_size).copy()
        
        all_chunks = []
        for idx, row in sample_df.iterrows():
            text_to_chunk = row['searchable_content']
            
            # Select the chunking method based on the strategy name
            if strategy_name == 'sentence':
                chunks = self.sentence_based_chunking(text_to_chunk, **kwargs)
            elif strategy_name == 'fixed_length':
                chunks = self.fixed_length_chunking(text_to_chunk, **kwargs)
            elif strategy_name == 'semantic':
                chunks = self.semantic_chunking(text_to_chunk, **kwargs)
            else:
                chunks = [text_to_chunk] # No chunking, just use the whole text
            
            for chunk_idx, chunk_text in enumerate(chunks):
                all_chunks.append({
                    'paper_id': idx,
                    'chunk_id': chunk_idx,
                    'chunk_text': chunk_text,
                    'chunk_length': len(chunk_text),
                    'word_count': len(chunk_text.split())
                })
        
        chunks_df = pd.DataFrame(all_chunks)
        self.chunking_results[strategy_name] = chunks_df
        print(f"Created {len(chunks_df)} chunks from {sample_size} documents using '{strategy_name}' strategy.")
        return chunks_df
    
    def compare_chunking_strategies(self):
        """Generates and prints a comparison table of the applied strategies.
        This helps us pick the best strategy based on data.
        """
        if not self.chunking_results:
            print("No chunking strategies have been applied yet. Run 'apply_chunking_strategy' first.")
            return None
        
        comparison_data = {}
        for strategy, chunks_df in self.chunking_results.items():
            comparison_data[strategy] = {
                'Total Chunks': len(chunks_df),
                'Avg Chars/Chunk': chunks_df['chunk_length'].mean(),
                'Std Dev (Chars)': chunks_df['chunk_length'].std(),
                'Avg Words/Chunk': chunks_df['word_count'].mean(),
                'Min Chars/Chunk': chunks_df['chunk_length'].min(),
                'Max Chars/Chunk': chunks_df['chunk_length'].max()
            }
        
        comparison_df = pd.DataFrame(comparison_data).T
        print("\n--- Chunking Strategy Comparison ---")
        print(comparison_df.round(1))
        print("------------------------------------")
        return comparison_df

In [17]:
# Let's analyze different chunking strategies.
chunking_analyzer = ChunkingStrategyAnalyzer(healthcare_df)

# Apply each strategy with some reasonable parameters.
sentence_chunks = chunking_analyzer.apply_chunking_strategy('sentence', sentences_per_chunk=3)
fixed_length_chunks = chunking_analyzer.apply_chunking_strategy('fixed_length', chunk_size=400, overlap=50)
semantic_chunks = chunking_analyzer.apply_chunking_strategy('semantic', max_chunk_size=450)

# Now, let's see the results.
strategy_comparison = chunking_analyzer.compare_chunking_strategies()

Applying 'sentence' chunking strategy...
Created 6675 chunks from 1000 documents using 'sentence' strategy.
Applying 'fixed_length' chunking strategy...
Created 6414 chunks from 1000 documents using 'fixed_length' strategy.
Applying 'semantic' chunking strategy...
Created 5906 chunks from 1000 documents using 'semantic' strategy.

--- Chunking Strategy Comparison ---
              Total Chunks  Avg Chars/Chunk  Std Dev (Chars)  Avg Words/Chunk  \
sentence            6675.0            310.6            186.6             44.5   
fixed_length        6414.0            365.0             88.6             52.1   
semantic            5906.0            349.0             86.5             50.3   

              Min Chars/Chunk  Max Chars/Chunk  
sentence                  2.0           1125.0  
fixed_length              1.0            400.0  
semantic                  2.0            886.0  
------------------------------------


In [18]:
# Based on the analysis, semantic chunking provides a good balance of size and consistency.
# It avoids the extreme length variations of sentence chunking and the context-breaking nature of fixed-length chunking.
optimal_chunking_strategy = 'semantic'
print(f"\nSelected optimal chunking strategy: '{optimal_chunking_strategy}'")

# We'll use this choice for the entire dataset.
final_chunks_df = chunking_analyzer.apply_chunking_strategy(
    strategy_name=optimal_chunking_strategy,
    sample_size=len(healthcare_df), # Process the full dataset now
    max_chunk_size=450
)


Selected optimal chunking strategy: 'semantic'
Applying 'semantic' chunking strategy...
Created 321652 chunks from 60380 documents using 'semantic' strategy.


### =============================================================================
# SECTION 5: EMBEDDING LAYER IMPLEMENTATION
### =============================================================================

This is the final step of the **Embedding Layer**. Now that we have clean, well-sized chunks of text, we need to convert them into numerical vectors (embeddings). This process is what allows us to perform semantic search.

In [21]:
class MedicalEmbeddingSystem:
    """Handles the generation of text embeddings using a domain-specific model.
    
    We're choosing PubMedBERT because it's specifically trained on medical literature,
    making it much better at understanding the nuances of our data than a generic model.
    """
    
    def __init__(self):
        self.embedding_model = None
        self.embeddings = None
        self._load_embedding_model() # Underscore indicates a private helper method
    
    def _load_embedding_model(self):
        """Loads the best available embedding model for our medical text.
        
        It tries to load the best model first (PubMedBERT) and has fallbacks for robustness.
        """
        print("Loading medical domain embedding model...")
        
        model_name = 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext' # Our top choice
        fallback_model = 'sentence-transformers/all-MiniLM-L6-v2' # A good, fast alternative
        
        try:
            # Load the best model
            self.embedding_model = SentenceTransformer(model_name)
            print(f"Successfully loaded specialized model: {model_name}")
        except Exception as e:
            print(f"Failed to load PubMedBERT ({e}). Loading fallback model...")
            try:
                self.embedding_model = SentenceTransformer(fallback_model)
                print(f"Successfully loaded fallback model: {fallback_model}")
            except Exception as e_fallback:
                raise Exception(f"Could not load any embedding model. Error: {e_fallback}")

        # A quick test to confirm the model is working and to see its output dimension.
        test_embedding = self.embedding_model.encode(["Cardiovascular disease prevention"])
        print(f"Model loaded. Embedding dimension: {len(test_embedding[0])}")
    
    def generate_embeddings(self, texts, batch_size=32):
        """Generates embeddings for a list of text chunks.
        
        We process in batches to efficiently use memory, especially with large datasets.
        Normalizing embeddings is important for using cosine similarity later.
        """
        print(f"Generating embeddings for {len(texts)} text chunks...")
        
        # The encode method from SentenceTransformer is highly optimized for this.
        self.embeddings = self.embedding_model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True # Crucial for efficient similarity search with FAISS
        )
        
        print(f"Generated embeddings with shape: {self.embeddings.shape}")
        return self.embeddings
    
    def save_embeddings(self, filename):
        """Saves the generated embeddings to a file.
        This is a huge time-saver so we don't have to re-generate them every time we run the notebook.
        """
        if self.embeddings is not None:
            np.save(filename, self.embeddings)
            print(f"Embeddings saved to '{filename}'")

In [22]:
# Initialize the embedding system and generate embeddings for our final chunks.
embedding_system = MedicalEmbeddingSystem()
document_embeddings = embedding_system.generate_embeddings(final_chunks_df['chunk_text'].tolist())

# Save the embeddings so we can quickly load them next time.
embedding_filename = f"healthcare_embeddings_pubmedbert_{datetime.now().strftime('%Y%m%d')}.npy"
embedding_system.save_embeddings(embedding_filename)

Loading medical domain embedding model...


No sentence-transformers model found with name microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext. Creating a new one with mean pooling.


Successfully loaded specialized model: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Model loaded. Embedding dimension: 768
Generating embeddings for 321652 text chunks...


Batches:   0%|          | 0/10052 [00:00<?, ?it/s]

Generated embeddings with shape: (321652, 768)
Embeddings saved to 'healthcare_embeddings_pubmedbert_20250829.npy'


### =============================================================================
# SECTION 6: SEARCH LAYER WITH CACHING AND RE-RANKING
### =============================================================================

With our documents converted into vectors, we can now build the **Search Layer**. This layer's job is to take a user's question, convert it into a vector, and then find the most similar document vectors in our database. We'll add caching and re-ranking to make it fast and accurate.

In [25]:
class AdvancedSearchSystem:
    """Implements a two-stage vector search system with caching and re-ranking.
    
    Stage 1: Fast retrieval using FAISS.
    Stage 2: Accurate re-ranking using a CrossEncoder model.
    """
    
    def __init__(self, dataframe, chunks_df, embeddings, embedding_model):
        self.df = dataframe
        self.chunks_df = chunks_df
        self.embeddings = embeddings
        self.embedding_model = embedding_model
        self.text_processor = MedicalTextProcessor()
        
        # For caching queries to speed up repeated searches
        self.query_cache = {}
        
        # Initialize the FAISS index for fast retrieval
        self.index = self._build_vector_index()
        
        # Load the CrossEncoder model for the re-ranking step
        self.reranker = self._load_reranker()

    def _build_vector_index(self):
        """Builds the FAISS index from our document embeddings."""
        print("Building FAISS index for fast search...")
        dimension = self.embeddings.shape[1]
        # IndexFlatIP is perfect for cosine similarity when vectors are normalized.
        index = faiss.IndexFlatIP(dimension)
        index.add(self.embeddings.astype('float32')) # FAISS requires float32
        print(f"FAISS index built with {index.ntotal} vectors.")
        return index

    def _load_reranker(self):
        """Loads the CrossEncoder model for re-ranking."""
        print("Loading re-ranking model...")
        try:
            # This model is lightweight but very effective for refining search results.
            reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L12-v2')
            print("Re-ranking model loaded successfully.")
            return reranker
        except Exception as e:
            print(f"Warning: Could not load re-ranking model ({e}). Proceeding without re-ranking.")
            return None

    def search(self, query, top_k=20):
        """Performs the initial, fast search using the FAISS index."""
        # First, process the user's query just like we processed the documents.
        processed_query = self.text_processor.clean_medical_text(query)
        query_embedding = self.embedding_model.encode([processed_query], normalize_embeddings=True)
        
        # Search the FAISS index to get the top_k most similar chunk indices.
        scores, indices = self.index.search(query_embedding.astype('float32'), top_k)
        
        # Map the indices back to our original document chunks.
        results = []
        for score, idx in zip(scores[0], indices[0]):
            chunk_info = self.chunks_df.iloc[idx]
            original_paper = self.df.iloc[chunk_info['paper_id']]
            results.append({
                'similarity_score': score,
                'chunk_text': chunk_info['chunk_text'],
                'title': original_paper['title'],
                'abstract': original_paper['abstract'], # We'll need this for the final prompt
                'journal': original_paper['journal'],
                'condition_category': original_paper['condition_category']
            })
        return results
    
    def rerank(self, query, initial_results, final_k=5):
        """Re-ranks the initial search results for higher accuracy."""
        if not self.reranker or not initial_results:
            return initial_results[:final_k]
        
        # The cross-encoder needs pairs of (query, document_text).
        pairs = [(query, result['chunk_text']) for result in initial_results]
        
        # Predict gives a more accurate relevance score for each pair.
        rerank_scores = self.reranker.predict(pairs)
        
        # Add the new scores and sort to get the best results.
        for result, score in zip(initial_results, rerank_scores):
            result['rerank_score'] = score
        
        reranked_results = sorted(initial_results, key=lambda x: x['rerank_score'], reverse=True)
        return reranked_results[:final_k]

    def comprehensive_search(self, query, initial_k=20, final_k=5):
        """The main search function that combines retrieval, caching, and re-ranking."""
        cache_key = f"{query}_{initial_k}_{final_k}"
        if cache_key in self.query_cache:
            print("(Retrieved from cache)")
            return self.query_cache[cache_key]
        
        # Step 1: Fast retrieval
        initial_results = self.search(query, top_k=initial_k)
        
        # Step 2: Accurate re-ranking
        final_results = self.rerank(query, initial_results, final_k=final_k)
        
        # Store in cache for next time
        self.query_cache[cache_key] = final_results
        
        print(f"Retrieved and re-ranked top {len(final_results)} results.")
        return final_results

In [26]:
# Initialize the advanced search system with all our components.
search_system = AdvancedSearchSystem(
    dataframe=healthcare_df,
    chunks_df=final_chunks_df,
    embeddings=document_embeddings,
    embedding_model=embedding_system.embedding_model
)

print("Advanced search system is ready.")

Building FAISS index for fast search...
FAISS index built with 321652 vectors.
Loading re-ranking model...
Re-ranking model loaded successfully.
Advanced search system is ready.


### =============================================================================
# SECTION 7: GENERATION LAYER WITH MEDICAL PROMPTING
### =============================================================================

This is the **Generation Layer**. Its job is to take the high-quality context we've retrieved and use a powerful Large Language Model (LLM) like Gemini to generate a final, evidence-based answer. The key here is crafting a really good prompt.

In [70]:
class MedicalResponseGenerator:
    """Generates human-like, evidence-based answers using the Gemini model and retrieved context.
    
    The prompt engineering here is critical to ensure the answers are reliable and safe.
    """
    
    def __init__(self, gemini_model):
        self.model = gemini_model
        self.conversation_history = []
    
    def create_medical_prompt(self, query, search_results, max_context_papers=3):
        """Constructs a detailed, instruction-based prompt for the LLM.
        
        It dynamically changes based on whether relevant documents were found.
        """
        context_papers = search_results[:max_context_papers]
        
        # Format the retrieved papers into a readable context block.
        context_text = ""
        if context_papers:
            for i, paper in enumerate(context_papers, 1):
                context_text += f"\n--- Research Paper {i} ---\n"
                context_text += f"Title: {paper['title']}\n"
                context_text += f"Abstract: {paper['abstract']}\n"
                context_text += f"Journal: {paper['journal']}\n"
        
        # This is our main prompt template. It's designed to be robust.
        if context_text:
            prompt = f"""
            **Role:** You are a highly knowledgeable medical AI assistant. Your purpose is to provide clear, evidence-based answers to medical questions based on the research provided.
            
            **User's Query:** "{query}"
            
            **Relevant Research Evidence:**
            {context_text}
            
            **Your Task & Instructions:**
            1.  **Synthesize an Answer:** Create a comprehensive, well-structured answer to the user's query.
            2.  **Evidence-Based:** Your answer MUST be primarily based on the information in the 'Relevant Research Evidence' section above.
            3.  **Cite Your Sources:** When you use information from a paper, explicitly mention it (e.g., "According to Research Paper 1...").
            4.  **Acknowledge Limits:** If the evidence is inconclusive or doesn't fully answer the question, state that clearly.
            5.  **Clarity is Key:** Explain complex medical terms in simple language.
            6.  **DO NOT Hallucinate:** If the provided text does not contain the answer, state that the information is not available in the provided context.
            7.  **Crucial Disclaimer:** ALWAYS end your response with the following disclaimer: "This information is for educational purposes only and should not replace professional medical advice."
            
            **Answer:**

            **References:**
            - It should be the top 3 Research Papers
            - It must in this Format (all are provided in the data)
                - Research Papers Name
                - Journal
                - URL
                
            example - 
            ### References
            1. Major depression, anxiety disorder and suicidality in epilepsy: What should neurologists do?
               Journal: Epilepsy & behavior reports
               URL: https://pubmed.ncbi.nlm.nih.gov/40162063/

            2. Pain Management in the Elderly: A Narrative Review.
               Journal: Anesthesiology clinics
               URL: https://pubmed.ncbi.nlm.nih.gov/37516502/
               ...
            """
        else:
            # This is the fallback prompt if our search returns nothing relevant.
            prompt = f"""
            **Role:** You are a helpful medical AI assistant.
            
            **User's Query:** "{query}"
            
            **Your Task & Instructions:**
            1.  **Acknowledge No Findings:** Start by clearly stating that no directly relevant research papers were found in the database for this specific query.
            2.  **Provide General Knowledge:** Based on your general medical training, provide a helpful, general overview of the topic.
            3.  **Avoid Making definitive claims:** Since you don't have specific studies to back you up, use cautious language (e.g., "Generally, it is thought...", "One common approach is...").
            4.  **Crucial Disclaimer:** ALWAYS end your response with the following disclaimer: "This information is for educational purposes only and should not replace professional medical advice. No specific studies were found in the database to support this response."
            
            **Answer:**
            """
        return prompt
    
    def generate_response(self, query, search_results):
        """Generates the final response by calling the Gemini API with our crafted prompt."""
        print("Generating evidence-based response...")
        try:
            prompt = self.create_medical_prompt(query, search_results)
            
            # The actual call to the Gemini model
            response = self.model.generate_content(prompt)
            generated_text = response.text
            
            # Log this interaction for potential review later
            self.conversation_history.append({'query': query, 'response': generated_text})
            
            return generated_text
        except Exception as e:
            print(f"Error during response generation: {e}")
            return "I'm sorry, but I encountered an error while generating a response. Please try again."

In [72]:
# Initialize our response generator with the Gemini model we configured earlier.
response_generator = MedicalResponseGenerator(gemini_model)
print("Medical response generator is ready.")

Medical response generator is ready.


### =============================================================================
# SECTION 8: COMPLETE RAG SYSTEM ORCHESTRATION
### =============================================================================

Now we bring it all together. This class will act as the main controller, orchestrating the flow from user query through the search layer and finally to the generation layer.

In [76]:
class HealthcareRAGSystem:
    """The main orchestrator for the entire RAG pipeline.
    
    This class connects the Search and Generation layers to provide an end-to-end solution.
    """
    
    def __init__(self, search_system, response_generator):
        self.search_system = search_system
        self.response_generator = response_generator
        self.query_log = []
    
    def ask(self, query, detailed_results=False):
        """Processes a medical query through the full RAG pipeline and displays the output."""
        print(f"\n{'='*60}")
        print(f"Processing Query: '{query}'")
        start_time = time.time()
        
        # 1. Search for relevant documents (retrieval + re-ranking)
        print("Step 1: Searching medical literature...")
        search_results = self.search_system.comprehensive_search(query)
        
        # 2. Generate a response based on the search results
        print("\nStep 2: Synthesizing the answer...")
        response = self.response_generator.generate_response(query, search_results)
        
        processing_time = time.time() - start_time
        
        # Log the full interaction for analysis
        result_log = {
            'query': query,
            'response': response,
            'search_results': search_results,
            'processing_time': processing_time
        }
        self.query_log.append(result_log)
        
        # Display the final output in a clean format
        self._display_results(result_log, detailed=detailed_results)
        
        return result_log
    
    def _display_results(self, result, detailed=False):
        """Helper function to print the results in a readable format."""
        print("\n--- AI Generated Response ---")
        print(result['response'])
        print("---------------------------")
        
        if detailed and result['search_results']:
            print("\n--- Top Retrieved Documents ---")
            for i, paper in enumerate(result['search_results'], 1):
                print(f"[{i}] Score: {paper.get('rerank_score', paper['similarity_score']):.4f}")
                print(f"    Title: {paper['title']}")
                print(f"    Chunk: {paper['chunk_text'][:150]}...")
            print("-----------------------------")
        
        print(f"\n[Processing time: {result['processing_time']:.2f} seconds]")
        print(f"{'='*60}\n")

In [78]:
# Final assembly! Let's create our complete RAG system.
healthcare_rag = HealthcareRAGSystem(search_system, response_generator)
print("Complete Healthcare RAG system initialized and ready to answer questions.")

Complete Healthcare RAG system initialized and ready to answer questions.


### =============================================================================
# SECTION 9: PROJECT EVALUATION WITH REQUIRED QUERIES
### =============================================================================

To properly evaluate the system, we'll test it against three challenging, self-designed queries. This will demonstrate the end-to-end capability of the system.

In [82]:
# These are the three official evaluation queries for the project.
evaluation_queries = [
    "What are the latest treatment guidelines for type 2 diabetes management in primary care?",
    "How should major depressive disorder be treated in elderly patients with multiple comorbidities?",
    "What are the current evidence-based protocols for acute myocardial infarction management in emergency departments?"
]

# Let's run each query through our system.
for query in evaluation_queries:
    # We'll set detailed_results=True to see the context the AI used.
    healthcare_rag.ask(query, detailed_results=True)


Processing Query: 'What are the latest treatment guidelines for type 2 diabetes management in primary care?'
Step 1: Searching medical literature...
(Retrieved from cache)

Step 2: Synthesizing the answer...
Generating evidence-based response...

--- AI Generated Response ---
Based on the provided research, here is a summary of the approach to managing type 2 diabetes.

The latest guidelines for managing type 2 diabetes emphasize a holistic and comprehensive approach. According to Research Papers 1, 2, and 3, healthcare professionals are encouraged to view type 2 diabetes not just as a blood sugar issue, but as a **cardiorenal metabolic syndrome**. This means it is a condition that affects the heart, kidneys, and the body's metabolism.

The core principles of management outlined in the research include:

*   **Holistic Management:** Treatment should not focus solely on blood glucose. Instead, it requires the simultaneous management of:
    *   Blood glucose
    *   Blood pressure
    

### =============================================================================
# SECTION 10: INTERACTIVE TESTING SESSION
### =============================================================================

Now for the fun part! This section provides an interactive loop where you can ask any medical question you want and see the RAG system in action.

In [84]:
def interactive_session():
    """Creates an interactive command-line interface to chat with the RAG system."""
    print("\n--- Interactive Medical Query Session ---")
    print("Type your question and press Enter. Type 'quit' to exit.")
    
    while True:
        query = input("\nYour Query: ")
        if query.lower() == 'quit':
            print("Exiting interactive session. Goodbye!")
            break
        if not query.strip():
            continue
        
        # Use the main RAG system to get an answer
        healthcare_rag.ask(query, detailed_results=False) # Keep it compact for chat
        
# To start the interactive session, uncomment and run the line below.

interactive_session()


--- Interactive Medical Query Session ---
Type your question and press Enter. Type 'quit' to exit.



Your Query:  What are the contraindications for beta-blockers in cardiovascular disease



Processing Query: 'What are the contraindications for beta-blockers in cardiovascular disease'
Step 1: Searching medical literature...
Retrieved and re-ranked top 5 results.

Step 2: Synthesizing the answer...
Generating evidence-based response...

--- AI Generated Response ---
Based on the research provided, here are the key contraindications and factors that limit the use of beta-blockers in cardiovascular disease.

### Summary of Findings

The provided research highlights that while beta-blockers are essential for treating many cardiovascular conditions, their use can be limited by specific clinical situations and side effects. A key contraindication is initiating therapy during an acute heart failure episode, and common side effects like bradycardia and hypotension often prevent patients from reaching optimal doses. Conversely, a common historical contraindication, Chronic Obstructive Pulmonary Disease (COPD), is no longer considered an absolute reason to avoid certain beta-blocke


Your Query:  How should major depressive disorder be treated in elderly patients with multiple comorbidities?



Processing Query: 'How should major depressive disorder be treated in elderly patients with multiple comorbidities?'
Step 1: Searching medical literature...
(Retrieved from cache)

Step 2: Synthesizing the answer...
Generating evidence-based response...

--- AI Generated Response ---
Based on the research provided, treating major depressive disorder (MDD) in elderly patients with multiple comorbidities is a complex issue where treatment effectiveness can be significantly influenced by the specific co-occurring conditions.

### Synthesis of Findings

**Impact of Physical Comorbidities on Treatment Outcomes**

The presence of comorbid physical health problems can directly affect how quickly a patient responds to pharmacological treatment for depression. According to Research Paper 3, which studied patients with psychotic depression (a severe form of MDD), a "higher burden of comorbid physical problems" was an independent predictor of a *longer* time to achieve remission. This suggests t


Your Query:  How is chronic kidney disease diagnosed and staged?



Processing Query: 'How is chronic kidney disease diagnosed and staged?'
Step 1: Searching medical literature...
Retrieved and re-ranked top 5 results.

Step 2: Synthesizing the answer...
Generating evidence-based response...

--- AI Generated Response ---
Based on the research provided, here is a summary of how chronic kidney disease (CKD) is diagnosed and staged.

### Diagnosis of Chronic Kidney Disease

The diagnosis of chronic kidney disease relies on specific laboratory tests, as the condition is often asymptomatic in its early stages (Research Paper 2, Research Paper 3). Screening is recommended, particularly for high-risk populations like individuals with type 2 diabetes (Research Paper 1).

According to the provided evidence, the key diagnostic tests are:

1.  **Serum Creatinine and Estimated Glomerular Filtration Rate (eGFR):** Diagnosis involves a blood test to measure creatinine levels. Creatinine is a waste product that healthy kidneys filter out of the blood. This measurem


Your Query:  quit


Exiting interactive session. Goodbye!


### =============================================================================
# SECTION 11: SCREENSHOT-READY OUTPUT FOR SUBMISSION
### =============================================================================
- This section generates structured outputs in 6 blocks for project submission screenshots as required by the assignment guidelines.


In [86]:
import time
class SubmissionOutputGenerator:
    """Generates structured outputs for project submission screenshots."""
    
    def __init__(self, rag_system):
        self.rag_system = rag_system
        self.submission_queries = [
            "What are the latest treatment guidelines for type 2 diabetes management in primary care?",
            "How should major depressive disorder be treated in elderly patients with multiple comorbidities?",
            "What are the current evidence-based protocols for acute myocardial infarction management in emergency departments?"
        ]
    
    def display_block_header(self, block_num, query_num, content_type):
        """Displays a clear block header for screenshots."""
        print("\n" + "="*100)
        print(f"SCREENSHOT BLOCK {block_num}: QUERY {query_num} - {content_type.upper()}")
        print("="*100)
    
    def display_search_results_block(self, query, query_num, block_num):
        """Displays the top 3 search results for a query in a screenshot-ready format."""
        self.display_block_header(block_num, query_num, "TOP 3 RETRIEVED RESULTS")
        
        print(f"\nQUERY: {query}\n")
        print("-" * 80)
        
        # Get search results from the system
        search_results = self.rag_system.search_system.comprehensive_search(query, final_k=3)
        
        for i, result in enumerate(search_results, 1):
            print(f"\nDOCUMENT {i}:")
            print(f"   Title: {result['title']}")
            print(f"   Journal: {result['journal']}")
            print(f"   Similarity Score: {result.get('rerank_score', result['similarity_score']):.4f}")
            print(f"   Content Excerpt: {result['chunk_text'][:300]}...")
            print(f"   Medical Category: {result['condition_category']}")
            print("-" * 80)
        
        print("\nEND OF BLOCK - READY FOR SCREENSHOT")
        print("="*100)
    
    def display_generated_answer_block(self, query, query_num, block_num):
        """Displays the AI-generated answer for a query in a screenshot-ready format."""
        self.display_block_header(block_num, query_num, "FINAL GENERATED ANSWER")
        
        print(f"\nQUERY: {query}\n")
        print("-" * 80)
        
        # Get the complete response from the system
        search_results = self.rag_system.search_system.comprehensive_search(query)
        generated_response = self.rag_system.response_generator.generate_response(query, search_results)
        
        print("\nAI-GENERATED RESPONSE:")
        print("\n" + generated_response)
        
        print("\nEND OF BLOCK - READY FOR SCREENSHOT")
        print("="*100)

# Initialize the submission output generator
submission_generator = SubmissionOutputGenerator(healthcare_rag)
print("Submission output generator ready!")

# =============================================================================
# GENERATE INDIVIDUAL BLOCKS FOR SCREENSHOTS
# =============================================================================

print("\n" + "="*80)
print("INSTRUCTIONS FOR SCREENSHOT CAPTURE")
print("="*80)
print("Run each block below one by one and take a screenshot after each:")
print("1. Block 1: Query 1 - Search Results")
print("2. Block 2: Query 1 - Generated Answer") 
print("3. Block 3: Query 2 - Search Results")
print("4. Block 4: Query 2 - Generated Answer")
print("5. Block 5: Query 3 - Search Results") 
print("6. Block 6: Query 3 - Generated Answer")
print("="*80)

# BLOCK 1: Query 1 - Search Results
print("\nGenerating BLOCK 1...")
submission_generator.display_search_results_block(
    "What are the latest treatment guidelines for type 2 diabetes management in primary care?", 1, 1
)

input("\nPRESS ENTER after taking screenshot of Block 1 to continue...")

# BLOCK 2: Query 1 - Generated Answer
print("\nGenerating BLOCK 2...")
submission_generator.display_generated_answer_block(
    "What are the latest treatment guidelines for type 2 diabetes management in primary care?", 1, 2
)

input("\nPRESS ENTER after taking screenshot of Block 2 to continue...")

# BLOCK 3: Query 2 - Search Results
print("\nGenerating BLOCK 3...")
submission_generator.display_search_results_block(
    "How should major depressive disorder be treated in elderly patients with multiple comorbidities?", 2, 3
)

input("\nPRESS ENTER after taking screenshot of Block 3 to continue...")

# BLOCK 4: Query 2 - Generated Answer
print("\nGenerating BLOCK 4...")
submission_generator.display_generated_answer_block(
    "How should major depressive disorder be treated in elderly patients with multiple comorbidities?", 2, 4
)

input("\nPRESS ENTER after taking screenshot of Block 4 to continue...")

# BLOCK 5: Query 3 - Search Results
print("\nGenerating BLOCK 5...")
submission_generator.display_search_results_block(
    "What are the current evidence-based protocols for acute myocardial infarction management in emergency departments?", 3, 5
)

input("\nPRESS ENTER after taking screenshot of Block 5 to continue...")

# BLOCK 6: Query 3 - Generated Answer
print("\nGenerating BLOCK 6...")
submission_generator.display_generated_answer_block(
    "What are the current evidence-based protocols for acute myocardial infarction management in emergency departments?", 3, 6
)

print("\n" + "="*80)
print("ALL 6 SCREENSHOT BLOCKS COMPLETED!")
print("="*80)
print("You now have all required screenshots for your submission:")
print("- 3 Search Results blocks (Blocks 1, 3, 5)")
print("- 3 Generated Answer blocks (Blocks 2, 4, 6)")
print("="*80)

Submission output generator ready!

INSTRUCTIONS FOR SCREENSHOT CAPTURE
Run each block below one by one and take a screenshot after each:
1. Block 1: Query 1 - Search Results
2. Block 2: Query 1 - Generated Answer
3. Block 3: Query 2 - Search Results
4. Block 4: Query 2 - Generated Answer
5. Block 5: Query 3 - Search Results
6. Block 6: Query 3 - Generated Answer

Generating BLOCK 1...

SCREENSHOT BLOCK 1: QUERY 1 - TOP 3 RETRIEVED RESULTS

QUERY: What are the latest treatment guidelines for type 2 diabetes management in primary care?

--------------------------------------------------------------------------------
Retrieved and re-ranked top 3 results.

DOCUMENT 1:
   Title: Type 2 diabetes: treatment recommendations for reducing the risk of complications.
   Journal: Nursing standard (Royal College of Nursing (Great Britain) : 1987)
   Similarity Score: 6.5557
   Content Excerpt: This article explains the latest national guidelines on managing type 2 diabetes in adults, including the


PRESS ENTER after taking screenshot of Block 1 to continue... 



Generating BLOCK 2...

SCREENSHOT BLOCK 2: QUERY 1 - FINAL GENERATED ANSWER

QUERY: What are the latest treatment guidelines for type 2 diabetes management in primary care?

--------------------------------------------------------------------------------
(Retrieved from cache)
Generating evidence-based response...

AI-GENERATED RESPONSE:

Based on the research evidence provided, here is a summary of the approach to managing type 2 diabetes in primary care.

Based on the provided research, the latest guidelines for type 2 diabetes management emphasize a holistic and comprehensive approach. According to Research Papers 1, 2, and 3, type 2 diabetes is now recognized as a **cardiorenal metabolic syndrome**. This means it is a complex condition that affects not only blood sugar but also the heart, kidneys, and the body's overall metabolism.

The core principles of management highlighted in the research include:

*   **Holistic Management:** The guidelines advocate for managing multiple fac


PRESS ENTER after taking screenshot of Block 2 to continue... 



Generating BLOCK 3...

SCREENSHOT BLOCK 3: QUERY 2 - TOP 3 RETRIEVED RESULTS

QUERY: How should major depressive disorder be treated in elderly patients with multiple comorbidities?

--------------------------------------------------------------------------------
Retrieved and re-ranked top 3 results.

DOCUMENT 1:
   Title: Major depression, anxiety disorder and suicidality in epilepsy: What should neurologists do?
   Journal: Epilepsy & behavior reports
   Similarity Score: 1.4878
   Content Excerpt: Major depression, anxiety disorder and suicidality in epilepsy: What should neurologists do Four to five patients with epilepsy (PWE) can suffer from Major Depressive episodes (MDE) Comorbid anxiety disorders (AD) frequently occur together with MDE Failure to treat MDE can negatively affect several ...
   Medical Category: psychiatry
--------------------------------------------------------------------------------

DOCUMENT 2:
   Title: Antidepressants for treating depression among older 

...


Generating BLOCK 4...

SCREENSHOT BLOCK 4: QUERY 2 - FINAL GENERATED ANSWER

QUERY: How should major depressive disorder be treated in elderly patients with multiple comorbidities?

--------------------------------------------------------------------------------
(Retrieved from cache)
Generating evidence-based response...

AI-GENERATED RESPONSE:

Based on the research provided, here is a synthesis of information regarding the treatment of major depressive disorder (MDD) in elderly patients with multiple comorbidities.

### Summary of Findings

Treating major depressive disorder in elderly patients with multiple health conditions is complex, and the effectiveness of treatment can be influenced by the specific comorbidities present. The provided research highlights challenges in treatment efficacy for certain conditions and identifies factors that may predict treatment outcomes.

#### Efficacy of Antidepressants in Specific Comorbidities

The evidence on the effectiveness of standard an


PRESS ENTER after taking screenshot of Block 4 to continue... 



Generating BLOCK 5...

SCREENSHOT BLOCK 5: QUERY 3 - TOP 3 RETRIEVED RESULTS

QUERY: What are the current evidence-based protocols for acute myocardial infarction management in emergency departments?

--------------------------------------------------------------------------------
Retrieved and re-ranked top 3 results.

DOCUMENT 1:
   Title: The 2023 protocol for update to acute treatment of adults with migraine in the emergency department: The American Headache Society evidence assessment of parenteral pharmacotherapies.
   Journal: Headache
   Similarity Score: 3.2342
   Content Excerpt: The 2023 protocol for update to acute treatment of adults with migraine in the emergency department: The American Headache Society evidence assessment of parenteral pharmacotherapies The primary objective of this proposed guideline is to update the prior 2016 guideline on parenteral pharmacotherapie...
   Medical Category: emergency_medicine
----------------------------------------------------------


PRESS ENTER after taking screenshot of Block 5 to continue... 



Generating BLOCK 6...

SCREENSHOT BLOCK 6: QUERY 3 - FINAL GENERATED ANSWER

QUERY: What are the current evidence-based protocols for acute myocardial infarction management in emergency departments?

--------------------------------------------------------------------------------
(Retrieved from cache)
Generating evidence-based response...

AI-GENERATED RESPONSE:

Based on the research evidence provided, I cannot answer your question about the current evidence-based protocols for acute myocardial infarction management in emergency departments.

The provided research papers do not contain information on this topic. Instead, they focus on the management of other acute conditions in the emergency department:

*   **Research Papers 1 and 2** describe a protocol to create an updated clinical guideline for the acute treatment of adults with **migraine** in the emergency department. They focus on evaluating injectable medications and nerve blocks for migraine relief.
*   **Research Paper 3**